# ConversaAI — Fine-tuning XLM-R Intención (9 clases)
## Google Colab

Fine-tunea `xlm-roberta-base` para 9 clases de intención en ES y PT.

## 1. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Instalar dependencias

In [ ]:
!pip install -q transformers datasets torch tqdm accelerate evaluate huggingface_hub pyarrow scikit-learn

## 3. Configurar entorno

In [ ]:
import os
import json
import torch
import numpy as np
import pandas as pd
from datetime import datetime

os.environ['HF_HOME'] = '/content/.cache/huggingface'
os.environ['HF_DATASETS_CACHE'] = '/content/.cache/datasets'
os.environ['TRANSFORMERS_CACHE'] = '/content/.cache/huggingface'

for d in ['/content/.cache/huggingface', '/content/.cache/datasets',
          '/content/logs', '/content/checkpoints']:
    os.makedirs(d, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 4. Configurar ruta de datos

Actualizá según dónde tengas los parquets en Drive.

In [ ]:
# CAMBIAR: ruta a la carpeta con train.parquet, val.parquet, test.parquet (intent)
DATA_PATH = '/content/drive/MyDrive/conversaai/intent_limpio'

print('Dataset:', DATA_PATH)
for f in os.listdir(DATA_PATH):
    print(' ', f)

## 5. Importaciones y login HF

In [ ]:
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed,
)
from sklearn.metrics import classification_report, confusion_matrix
from huggingface_hub import login, whoami
import evaluate

set_seed(42)

# Reemplazar con tu token de HF
HF_TOKEN = 'TU_TOKEN_HF_AQUI'
login(token=HF_TOKEN)
user_info = whoami()
print(f'Login exitoso: {user_info["name"]}')

HUB_MODEL_ID = 'rosaiselagonzalez/xlm-r-intent-espt'
print(f'Hub model ID: {HUB_MODEL_ID}')

## 6. Cargar datos

In [ ]:
df_train = pd.read_parquet(os.path.join(DATA_PATH, 'train.parquet'))
df_val = pd.read_parquet(os.path.join(DATA_PATH, 'val.parquet'))
df_test = pd.read_parquet(os.path.join(DATA_PATH, 'test.parquet'))

print(f'Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}')

# Taxonomy de 9 clases
taxonomy_path = os.path.join(DATA_PATH, 'taxonomy.json')
with open(taxonomy_path) as f:
    taxonomy = json.load(f)
id2label = {int(k): v for k, v in taxonomy.items()}
label2id = {v: int(k) for k, v in taxonomy.items()}
print(f'Taxonomy: {taxonomy}')

print('\nDistribucion (train):')
for lbl, cnt in df_train['label'].value_counts().sort_index().items():
    print(f'  {lbl} ({id2label[lbl]}): {cnt}')

dataset = DatasetDict({
    'train': Dataset.from_pandas(df_train[['text', 'label']]),
    'val': Dataset.from_pandas(df_val[['text', 'label']]),
    'test': Dataset.from_pandas(df_test[['text', 'label']]),
})

## 7. Tokenizer

In [ ]:
MODEL_NAME = 'xlm-roberta-base'
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(['text'])
tokenized_datasets = tokenized_datasets.rename_column('label', 'labels')
tokenized_datasets.set_format(
    type='torch', columns=['input_ids', 'attention_mask', 'labels']
)

print(f'Train: {len(tokenized_datasets["train"])}')
print(f'Val:   {len(tokenized_datasets["val"])}')
print(f'Test:  {len(tokenized_datasets["test"])}')

## 8. Modelo

XLM-R base con classification head nuevo de 9 clases.

In [ ]:
NUM_LABELS = 9

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
).to(device)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parametros: {total:,} total, {trainable:,} entrenables')

## 9. TrainingArguments

In [ ]:
training_args = TrainingArguments(
    output_dir='/content/checkpoints/intent',
    logging_dir='/content/logs/intent',
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=10,
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=torch.cuda.is_available(),
    fp16_full_eval=torch.cuda.is_available(),
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    greater_is_better=True,
    label_smoothing_factor=0.1,
    logging_steps=50,
    logging_first_step=True,
    report_to='none',
    push_to_hub=False,
    hub_model_id=HUB_MODEL_ID,
    seed=42,
    data_seed=42,
    dataloader_num_workers=2,
    remove_unused_columns=False,
)

print(f'batch_size: {training_args.per_device_train_batch_size}')
print(f'epochs: {training_args.num_train_epochs}')
print(f'fp16: {training_args.fp16}')

## 10. Métricas

In [ ]:
accuracy_metric = evaluate.load('accuracy')
f1_metric = evaluate.load('f1')

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    accuracy = accuracy_metric.compute(predictions=preds, references=labels)
    f1_weighted = f1_metric.compute(predictions=preds, references=labels, average='weighted')
    f1_macro = f1_metric.compute(predictions=preds, references=labels, average='macro')
    return {
        'accuracy': accuracy['accuracy'],
        'f1_weighted': f1_weighted['f1'],
        'f1_macro': f1_macro['f1'],
    }

## 11. Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['val'],
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2,
            early_stopping_threshold=0.001,
        )
    ],
)
print('Trainer creado con early stopping patience=2')

## 12. Entrenar

In [ ]:
print('=' * 60)
print('INICIANDO ENTRENAMIENTO')
print('=' * 60)
print(f'Train: {len(tokenized_datasets["train"])}')
print(f'Val:   {len(tokenized_datasets["val"])}')
print(f'Early stopping: patience=2')
print('=' * 60)

train_result = trainer.train()

train_metrics = train_result.metrics
print(f'\nMetricas finales:')
for k, v in train_metrics.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

## 13. Evaluar en test

In [ ]:
print('Evaluando en test...')
test_results = trainer.evaluate(eval_dataset=tokenized_datasets['test'])
print(f'Test accuracy: {test_results["eval_accuracy"]:.4f}')
print(f'Test f1_weighted: {test_results["eval_f1_weighted"]:.4f}')
print(f'Test f1_macro: {test_results["eval_f1_macro"]:.4f}')
print(f'Test loss: {test_results["eval_loss"]:.4f}')

predictions = trainer.predict(tokenized_datasets['test'])
preds = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

print('\nClassification Report:')
print(classification_report(
    true_labels, preds,
    target_names=[id2label[i] for i in range(NUM_LABELS)],
    digits=4
))

## 14. Push a Hugging Face Hub

In [ ]:
print(f'Pusheando modelo a HF Hub: {HUB_MODEL_ID}...')
trainer.push_to_hub()
tokenizer.push_to_hub(HUB_MODEL_ID)
print(f'Listo: https://huggingface.co/{HUB_MODEL_ID}')

## 15. Prueba rápida

In [ ]:
from transformers import pipeline

pipe = pipeline(
    'text-classification',
    model=HUB_MODEL_ID,
    tokenizer=MODEL_NAME,
    device=0 if torch.cuda.is_available() else -1,
)

tests = [
    'Quiero cancelar mi pedido',
    'Cual es el horario de atencion?',
    'Me llego la factura con un error',
    'Dejo una opinion sobre el servicio',
    'Necesito cambiar mi direccion',
    'Quiero modificar el color del producto',
    'Esto es una verguenza, pesimo servicio',
    'Solicito el reembolso de mi dinero',
    'Donde esta mi pedido?',
]

expected = [
    'cancelacion', 'consulta_general', 'facturacion_pago',
    'feedback', 'gestion_cuenta', 'modificacion_pedido',
    'queja', 'reembolso', 'seguimiento',
]

print(f"{'Texto':<55} {'Prediccion':<25} {'Esperado':<25} {'OK'}")
print('-' * 110)
for text, exp in zip(tests, expected):
    r = pipe(text[:512])[0]
    ok = 'OK' if r['label'] == exp else 'X'
    print(f'{text:<55} {r["label"]:<25} {exp:<25} {ok}')